# The implementation of the 2nd training pipeline, the "Reduced"
In this pipeline after initial normal preprocessing steps such as:

    1. Dropping duplicate rows
    2. Applying action mapping
    3. Applying port bucketing
    4. Applying log transformation to all numeric features
We then, before training, excluded features such as "Bytes" and "Packets" as these are a result of a summation between their respective *sent* and *received* components.

After training this CTGAN model the numerical features were inverse transformed (after initial scaling) and all derivable features were recalculated in both the real and artificial datasets.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import logging
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

In [2]:
# logging.basicConfig(level=logging.INFO)

In [3]:
root = Path.cwd().parent 
data_path = root / "Real Data" / "log2.csv"
df = pd.read_csv(data_path)

In [4]:
df = df.drop_duplicates()

action_mapping = {
    'allow': 0,
    'deny': 1,
    'drop': 2,
    'reset-both': 3
}
df['Action'] = df['Action'].map(action_mapping)

def bucket_port(port):
    if pd.isna(port):
        return 'unknown'
    if 0 <= port <= 1023:
        return 'well_known'
    elif 1024 <= port <= 49151:
        return 'registered'
    elif 49152 <= port <= 65535:
        return 'ephemeral'
    return 'other'

for col in ['Source Port', 'Destination Port', 'NAT Source Port', 'NAT Destination Port']:
    df[f'{col} Category'] = df[col].apply(bucket_port)

In [5]:
# df['Bytes_log'] = np.log1p(df['Bytes Sent'] + df['Bytes Received'])
df['Bytes Ratio'] = np.log1p(df['Bytes Sent'] / (df['Bytes Received'] + 1e-6))  

In [6]:
df['pkts_log'] = np.log1p(df['pkts_sent'] + df['pkts_received'])
df['pkts_ratio'] = np.log1p(df['pkts_sent'] / (df['pkts_received'] + 1e-6))

In [7]:
cols_to_scale = [
    'Bytes',
    'Bytes Sent',
    'Bytes Received',
    'Packets',
    'Elapsed Time (sec)',
    'pkts_sent',
    'pkts_received'
]

df[cols_to_scale] = df[cols_to_scale].apply(lambda x: np.log1p(x))

In [8]:
columns_to_exclude = ['Packets', 'Bytes', 'Bytes_log', 'Bytes Ratio', 'pkts_log', 'pkts_ratio'] # exclude derivable features, which are recalculated after training
df_for_ctgan = df.drop(columns=columns_to_exclude, errors='ignore')

In [9]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_for_ctgan)

INFO:sdv.metadata.single_table:Detected metadata:
INFO:sdv.metadata.single_table:{
    "columns": {
        "Source Port": {
            "sdtype": "numerical"
        },
        "Destination Port": {
            "sdtype": "numerical"
        },
        "NAT Source Port": {
            "sdtype": "numerical"
        },
        "NAT Destination Port": {
            "sdtype": "numerical"
        },
        "Action": {
            "sdtype": "categorical"
        },
        "Bytes Sent": {
            "sdtype": "numerical"
        },
        "Bytes Received": {
            "sdtype": "numerical"
        },
        "Elapsed Time (sec)": {
            "sdtype": "numerical"
        },
        "pkts_sent": {
            "sdtype": "numerical"
        },
        "pkts_received": {
            "sdtype": "numerical"
        },
        "Source Port Category": {
            "sdtype": "categorical"
        },
        "Destination Port Category": {
            "sdtype": "categorical"
        },
        "

In [10]:
synthesizer = CTGANSynthesizer(
    metadata=metadata,
    epochs=50,
    verbose=True
)

synthesizer.fit(df_for_ctgan)
synthesizer.save('Reduced.pkl')

C:\Users\Карен Оганян\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:144: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\Карен Оганян\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:122: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
INFO:SingleTableSynthesizer:{'EVENT': 'Instance', 'TIMESTAMP': datetime.datetime(2025, 6, 1, 20, 17, 0, 561679), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_6694a6637dc844a7aed0a733c1ee07ca'}
INFO:SingleTableSynthesizer:{'EVENT': 'Fit', 'TIMESTAMP': datetime.datetime(2025, 6, 1, 20, 17, 0, 562777), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_6694a6637dc844a7aed0a733c1ee07ca', 'TOTAL NUMBER OF T

In [26]:
synthetic_data = synthesizer.sample(len(df))

INFO:SingleTableSynthesizer:{'EVENT': 'Sample', 'TIMESTAMP': datetime.datetime(2025, 6, 1, 20, 42, 13, 509590), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_6694a6637dc844a7aed0a733c1ee07ca', 'TOTAL NUMBER OF TABLES': 1, 'TOTAL NUMBER OF ROWS': 57170, 'TOTAL NUMBER OF COLUMNS': 14}


In [27]:
inverse_cols = [
    'Bytes Sent',
    'Bytes Received',
    'Elapsed Time (sec)',
    'pkts_sent',
    'pkts_received'
]

synthetic_data[inverse_cols] = np.expm1(synthetic_data[inverse_cols])

synthetic_data[inverse_cols] = synthetic_data[inverse_cols].round().astype(int)

In [28]:
synthetic_data['Bytes'] = np.log1p(synthetic_data['Bytes Sent'] + synthetic_data['Bytes Received'])
synthetic_data['Packets'] = np.log1p(synthetic_data['pkts_sent'] + synthetic_data['pkts_received'])
# synthetic_data['Bytes_log'] = np.log1p(synthetic_data['Bytes'])
synthetic_data['pkts_log'] = np.log1p(synthetic_data['pkts_sent'] + synthetic_data['pkts_received'])

eps = 1e-6

synthetic_data['pkts_ratio'] = np.log1p(synthetic_data['pkts_sent'] / (synthetic_data['pkts_received'] + eps))
synthetic_data['Bytes Ratio'] = np.log1p(synthetic_data['Bytes Sent'] / (synthetic_data['Bytes Received'] + eps))

In [29]:
synthetic_data[inverse_cols] = synthetic_data[inverse_cols].apply(lambda x: np.log1p(x))

In [30]:
synthetic_data.to_csv('BackAndForth.csv')

In [31]:
df.to_csv('BackAndForthREAL.csv')